# LifeMate B043 Final — Active Graph & paired GPU evaluation

Run All: verify source/graph payload → CPU safety gates and actual B043-loop graph demos → Qwen3.5-4B non-thinking → paired GRAPH_DISABLED / GRAPH_ASSISTED_SAFE_READ → checkpoints, predictions, traces, report ZIP.

Attach the authorized BTC train dataset containing `zaic_tools` and `train_environment`. Enable GPU and Internet/model cache. The notebook embeds the final v3 graph and approved READ overlay. It embeds no raw train, environment files, credentials or GPU results. Every WRITE still comes from Decision Policy and passes B042 guards and ToolEnv.validate.

Local CPU tests use scripted Policy and fixture observations. They are not Qwen/ToolEnv accuracy measurements. The final GPU run has not been performed locally. State Match requires independent authoritative reference hashes; train partial final_state is not a full-state hash. Fuzzy-call F1 below is a Spec-oriented proxy, not official competition scoring.


In [ ]:
import os
from pathlib import Path
B043_GPU = os.environ.get('B043_GPU','auto')  # 0=CPU only, 1=require CUDA, auto=use CUDA if present
B043_MODES = ['GRAPH_DISABLED','GRAPH_ASSISTED_SAFE_READ']
B043_CASE_IDS = None  # automatic seeded representative 40 cases, beginning with the 5 reported regressions
B043_SAMPLE_LIMIT = 40
B043_GRAPH_DIR = os.environ.get('B043_GRAPH_DIR')  # optional dataset path ending in artifacts_final
B043_REFERENCE_HASHES = os.environ.get('B043_REFERENCE_HASHES')  # optional independent {sample_id: full_state_hash}
B043_BASE = Path(os.environ.get('B043_WORKDIR','/kaggle/working/lifemate_b043_final' if Path('/kaggle/working').is_dir() else str(Path.cwd()/'reports/notebook_cpu_run')))
BUNDLE_ROOT = B043_BASE/'bundle'
OUTPUT = B043_BASE/'outputs'
OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
import base64,hashlib,io,sys,zipfile,importlib
PAYLOAD_SHA256 = 'b696d3032d99f5790da925a609d285ec91afff4eee44516dd1a9858d9dcd9cfb'
EMBEDDED_SHA256 = {'approvals.example.json': '549134b50002aca8be8070eb2d53e6baf0b3b08acc95f0e3688e6866cfc6bb97', 'artifacts/approved_runtime_graph.json': 'a61e0d3d94150c50363129fa1d31f03189411ef898a1a18707e80f5aa2ec0326', 'artifacts/compiled_graph_v3.json': '67f7bf7ec9b5e196d7a50c32f2455ef3a79eece9fd57d147c21cb86aa9675b2d', 'artifacts/full_dependency_graph_v3.json': '37514613a9dfc9d74c5d7aa17a48ae982514a29208c96957cd16c4201bd826bd', 'artifacts/graph_manifest.json': 'd159ec4e2e8fd893acab4677d393e7beea95a0c298d1d705cf31407985fe353a', 'artifacts/graph_review_queue_v3.csv': 'c4b69318ebd2c43515163e8d81ac9190700ac46953c0fa6570bd403d9b2a3ee7', 'artifacts/graph_validation_report.json': '981dd6e242740fce36b2c0c18d4fb4f8c4d4bd973ba1c400ef18a843897e2079', 'artifacts/legacy_provenance_v3.json': 'bf5178c8e860fecfc4b605601aa5a4e03d877e79f60244f8831ac53f713c78e4', 'artifacts/source_integrity_report.json': '77090708e794bdb1c932cf1161dc8b5b84b17b1ecfc2f302d613726dbb7d571c', 'artifacts/tool_coverage.csv': 'acf09b9c15eed913192eef5f6dbbf99a1a1077de849fdbce44e93aa9c9fe7740', 'artifacts/tool_registry_audit.json': '9bbe1b012cec946f59e5cf74cbef9b3e39c18e337fdc0dea1f3911163e4e0cbd', 'artifacts_completed/candidate_completion.csv': '99d9e4fdb0a7574cda0ca86ac630028d045006341e70c0e3c30241fc152ff678', 'artifacts_completed/full_dependency_graph_v3.json': '7124d08739bbd3526903fb698c0dbe629ac00be9e09a549daa2ad74b3fb8f8bb', 'artifacts_completed/graph_manifest.json': 'b97c38fbd8f9ad23679b15332bd6dbc6a49e97e77e561ba7844bd8b81b507466', 'artifacts_completed/graph_validation_report.json': '7df05b2e1b621ec3f817e888888088f7e89fc47adf97316c3e8e8f4d4ee39b88', 'artifacts_completed/operator_approvals.json': '6f6befb5a682601a2d37f8296a8372298642b486bd8899f4f437cee5276ec33c', 'artifacts_final/approved_runtime_graph.json': 'dd4bab7dd23a496b8a438895e16061221cf8fde5cabd6eada6e43819d03037fa', 'artifacts_final/compile_report.json': 'bcd64bdd4f014ea93fd30e26a8abee9ada1ab98b1e1d05898e60ee0e24d826e3', 'artifacts_final/compiled_graph_v3.json': '908b021365f1d18af6266d84abc333e1b73bcc0b9f64f39de6469c8b97ca8635', 'artifacts_final/graph_manifest.json': '1a38efe337815a9ed0aef858790a1f293a07322a42051554e127dc7f76f306a1', 'artifacts_final/operator_approvals.json': '6f6befb5a682601a2d37f8296a8372298642b486bd8899f4f437cee5276ec33c', 'artifacts_reviewed/activation_report.json': 'cdf1af3c768208205524db612e106ffeefe9055aabe876dcdae80d08719c51c1', 'artifacts_reviewed/approved_runtime_graph.json': '420b2a6c90a8102f7f2542e284591a6044e572ab79af48f4a1af328ec8de7d71', 'artifacts_reviewed/B043_KAGGLE_ACTIVE_GRAPH_GUIDE.md': '6e63145919b3fc03b72de5f1fa5dd8610ebf7e98e1f0040f27a81af40b499374', 'artifacts_reviewed/B043_RUNTIME_ACTIVATION_REPORT.md': '141a5bad2710c1875e8fba224792af43b2c0d115c78bc1ac1555ba6a1b2af6a7', 'artifacts_reviewed/closed_loop_verification.json': 'c1bff362dfe776379bde87a2c0aa9c52137d15879d9e351a26ace1e11d69eedf', 'artifacts_reviewed/compile_report.json': '0a77a16e04984498979dcad1e6622799d2e5ce1fca5d48c6961fb119ce6e83ec', 'artifacts_reviewed/compiled_graph_v3.json': '728c99798c2600a88fad6e8fa4afb1141b47e206248307f2abb3106a4ad0d0a7', 'artifacts_reviewed/cpu/cpu_test_results.json': '8b88c481b9227a587c4f7c84c21726724ad6b3769d2e65d223a14c22e307e952', 'artifacts_reviewed/cpu/cpu_tests.txt': '9efe1479204df20e12125ece84c0084265dfaf64f61092ae61f3246bb1ce60b0', 'artifacts_reviewed/graph_manifest.json': 'edfe6a87edf56fbc8e0d89321e275e2380ca2c5c96f0e881717557bf3348f108', 'artifacts_reviewed/operator_approvals.json': 'd52aa19f511d499b7547772346dcdf15978865f944a2dee8b745df17ef4dd861', 'artifacts_reviewed/runtime_gate_results.json': 'b37cb86d6bad9522211320929321876ceb8b845b70b242309466e6adeb98a944', 'B043_FINAL_REPORT.md': 'd8d434eecd2ac63946e584d9c4ace5ee57550f59b67af23a601d959634e8eb0d', 'B043_KAGGLE_FINAL_GUIDE.md': 'e1e910244241ff7c47944c277d53f43ca6be130d986b9f6de1d7f0f4043df44c', 'inputs/LifeMate_B042_Progress_Retry_Controller_Kaggle.ipynb': '1e80fe50cab2eb195b665094d8f6560dd5215c09ce5dc001d56fb9403a6e00de', 'inputs/train_tools_catalog.json': 'a994984bb580d7e0d60166641858901d428549682ba3614dbf9269037a33cce0', 'lifemate_graph/__init__.py': 'aa61a0e3331711980c97561ad8cc3a00bcf7617fdc24c2ba84f91301fb0b11f0', 'lifemate_graph/b043.py': '3c8d85b0c5a9607a44125b1f46ee39a7db88b38e8e27d42622057f06eace65c6', 'lifemate_graph/builder.py': 'c5a8f41652f62011b4935581145a4a1fc8853ce5acd603ca29ec8f4e56f0eca8', 'lifemate_graph/cli.py': '6cbc8863b265f164fb19d0923463f3b2760e71393b4be687b1ca9dbc2df3ed0f', 'lifemate_graph/common.py': '7685041d3515820190cd79e841e07a3a8414c83e12df0f9eb24eee7f6f1cec42', 'lifemate_graph/compiler.py': '6d4cb5357aab67603acb90c9f5b66e303abc8da0e67772f48e18b72cf4e58452', 'lifemate_graph/context.py': '7a387bce080819aa2365a847b4f14ffecc2e9cb040b5060699da388b24e63ed9', 'lifemate_graph/demo.py': '923cbff62f666079c3ce63742cb8694cb69698fad920364d7bc57483ebd49961', 'lifemate_graph/evaluation.py': '717a2eac6eb4ee134e0c4220625ff8169666c4fc3b747b64e81d2867505d32a4', 'lifemate_graph/inference.py': '24652b9b5fae97fbadfcad8209e7ebfeac2ed3340b46ba1c7b7c92aa072ff983', 'lifemate_graph/kaggle.py': 'b2fc83fdf876e795f4c00d81f780d57991f06fa7d90c4b62e7c66c5ac45b1a9f', 'lifemate_graph/runtime.py': '263c2b83e8bebf96954a2f1c9f9d511fb20b2d061c5375eaac400e73d1c86269', 'lifemate_graph/validation.py': 'a3d7375668fd2d8a6054d0121c39b76f733edd2bc8e349f526136c5131d90d53', 'operator_approvals.json': 'd52aa19f511d499b7547772346dcdf15978865f944a2dee8b745df17ef4dd861', 'operator_approvals.proposed.json': '956faba8a4db54b48dbd7d830d69107839862b314596e92d573e81121abae253', 'reports/ready19_closed_loop.json': '987dfdea59d3c3470f319df29a56264bf00dc9b495c5d8061df2b7c92a9e90d8', 'requirements-kaggle.txt': '09c0aade9eabf6224d72044403a80d00d64b75d1a31caece576d4382d17b815d', 'runtime_activation_final.csv': '3a480b977b819bf17f6d9ac81ef13c7c375088a745d4f9db1415c5e2bd9f3cd4', 'runtime_approval_review.csv': 'a89ccecaa9471dd9d9aedef0d831ec94c70244a45b941b7cf2af8c964c2eab59', 'runtime_review/activation_manifest.json': '36188606a15704694bf323cc7af9ddd496cb80d2e33609c01a63cd78d7543f0b', 'runtime_review/binder_probes.json': 'eccd4c178b035fb5be9d0ad7af9847320b5515f387197c2ad57b7c551a269ec6', 'runtime_review/preapproval_checks.json': '1c375e580379d8290f82f0e3a0ef42293a68ce8368f6a3ad185245b4f4ba78e6', 'runtime_review/preapproval_checks.txt': '8b5c26513663f7f9fd087f8c54b6c8414fb8224f62391fc555c58303d3dee2cb', 'runtime_review/review_manifest.json': 'f06a3de80e6cdbc78a98c56fd60db3e5434302c3134b8366ec45828830c9a6fb', 'runtime_review/review_summary.json': '3cb3f4ca57c4a658821959089c2c736579baf363e1bba3b7abe6d97aaf9af4b7', 'schemas/dependency_graph_v3.schema.json': '783ee394acd27277554c1203976ef7419f26a26903e8986b265970ad9ca662e2', 'scripts/__init__.py': '143626520a19864f049e8e2eb70db72a810b40393afd70b3b736c9a48690bacf', 'scripts/prepare_runtime_activation.py': 'a60d5e81cbc6a454dd08bc0c8a68ae07ddda95c458ef7d2ee27164575436d707', 'scripts/review_runtime_approvals.py': '23b6d9522439e3c600e96d7adcd7c514ad17d357d1d9a68c3e4a6e3c3bd2aba1', 'scripts/run_cpu_checks.py': '5941acb7f7a7aa591e50d98b8a30fb3f9df844121e64a73fda78e24b91e4810c', 'tests/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'tests/data/approved_contracts.json': '947fcf8726f756ca029df396119bc29afc2d99f7ef1972746336b27607badfe0', 'tests/data/development_regressions.json': 'e353318d74d1e9b38dccd99c98548a69fca360254f4e6d9f2f35141a2075e1f8', 'tests/fixtures.py': '9eaf3349bbc95045905187cf9e153e1de87e25a12914fb3a6452c3e8cd98ed28', 'tests/ready_cases.py': '489983a9e3512b42fbc2f96286140dd01d40b124fd88a54c2d469391f6afcf9f', 'tests/test_budget.py': '12db453b457c8ce73a3876df6f7c53df44538982f95c59533822d46707cee63d', 'tests/test_build.py': 'f2da73c0ba3748599f60003ba2fbe92da6a12111b3ace82e314509134d0a2f45', 'tests/test_evaluation.py': 'd6f762c66996f34bb64a76cb20e0ab8765562f72b3eb2a44dbd658b75392c740', 'tests/test_fast_track.py': '5964869a14632aa45db1a4efd6415c6da39d40c7bef1e883226b61d759d4f85e', 'tests/test_final_graph.py': '1249d4017f23526488baf5c1505139e37f75648b482c7c1eb6890a9b38d3b489', 'tests/test_integration.py': '319853031054682429530ceb9b16f186dd457ee4200eef0c29e5f97d3cc4136b', 'tests/test_production_activation.py': '863ab5e338128b77bfc3ba269effa5ad331f30d184710968b76019d10df7813e', 'tests/test_resolver.py': 'fae5f442424412c52e2c180881474b09dc0248172b13a620c1a28d592e243787', 'tests/test_runtime_approval_review.py': 'f8486030496cae68eaab8237cd96eb891807fa2263cda0759fb979ccdbba9b5b'}
PAYLOAD = '...'
raw_bundle=base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw_bundle).hexdigest()==PAYLOAD_SHA256
BUNDLE_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw_bundle)) as archive:
    assert set(archive.namelist())==set(EMBEDDED_SHA256)
    for member in archive.infolist():
        target=(BUNDLE_ROOT/member.filename).resolve()
        if BUNDLE_ROOT.resolve() not in target.parents:raise ValueError('Unsafe payload member')
    archive.extractall(BUNDLE_ROOT)
for name,digest in EMBEDDED_SHA256.items():
    assert hashlib.sha256((BUNDLE_ROOT/name).read_bytes()).hexdigest()==digest, name
sys.path.insert(0,str(BUNDLE_ROOT))
for name in list(sys.modules):
    if any(name==p or name.startswith(p+'.') for p in ['lifemate_graph','tests','scripts']):del sys.modules[name]
importlib.invalidate_caches()
from lifemate_graph.common import read_json,write_json,sha256
from lifemate_graph.compiler import load_compiled
ACTIVE_GRAPH_DIR=Path(B043_GRAPH_DIR) if B043_GRAPH_DIR else BUNDLE_ROOT/'artifacts_final'
compiled=load_compiled(ACTIVE_GRAPH_DIR/'compiled_graph_v3.json',ACTIVE_GRAPH_DIR/'graph_manifest.json',BUNDLE_ROOT/'inputs/train_tools_catalog.json')
embedded=load_compiled(BUNDLE_ROOT/'artifacts_final/compiled_graph_v3.json',BUNDLE_ROOT/'artifacts_final/graph_manifest.json',BUNDLE_ROOT/'inputs/train_tools_catalog.json')
assert compiled['payload_sha256']==embedded['payload_sha256'],'External graph differs from the CPU-tested final graph'
manifest=read_json(ACTIVE_GRAPH_DIR/'graph_manifest.json')
for relative,digest in manifest['artifacts_sha256'].items():assert sha256(ACTIVE_GRAPH_DIR/relative)==digest,relative
authorized={d['edge_id'] for d in read_json(BUNDLE_ROOT/'artifacts_completed/operator_approvals.json')['decisions'] if d['decision']=='APPROVED'}
assert compiled['runtime_scope']=='PRODUCTION' and set(compiled['edges'])==authorized
print('Verified embedded files:',len(EMBEDDED_SHA256),'safe READ dependencies:',len(compiled['edges']))
print('Graph SHA:',compiled['graph_content_sha256'],'compiled payload SHA:',compiled['payload_sha256'])


In [ ]:
import runpy
cpu_runner=runpy.run_path(str(BUNDLE_ROOT/'scripts/run_cpu_checks.py'),run_name='b043_cpu_module')
CPU_REPORT=cpu_runner['run'](BUNDLE_ROOT,OUTPUT/'cpu')
assert CPU_REPORT['status']=='PASS','CPU regression failure: GPU stage blocked'
assert any('FinalGraphTests.test_all_approved_dependencies_participate_in_real_loop' in name for name in CPU_REPORT['passed'])
print('Final PRODUCTION graph passed CPU B043-loop gates. Fixture Policy/ToolEnv only.')


In [ ]:
import importlib.util
cuda=False
if B043_GPU!='0' and importlib.util.find_spec('torch'):
    import torch
    cuda=torch.cuda.is_available()
GPU_REPORT={'status':'NOT_RUN','gpu_executed':False,'reason':'CPU-only setting or no CUDA GPU'}
if B043_GPU=='1' and not cuda:GPU_REPORT={'status':'FAILED','gpu_executed':False,'reason':'GPU requested but CUDA unavailable'}
if B043_GPU!='0' and cuda:
    try:
        from lifemate_graph.kaggle import run_benchmark
        reference=read_json(B043_REFERENCE_HASHES) if B043_REFERENCE_HASHES else None
        GPU_REPORT=run_benchmark(BUNDLE_ROOT,OUTPUT/'benchmark',B043_CASE_IDS,B043_MODES,
            graph_dir=ACTIVE_GRAPH_DIR,reference_hashes=reference,limit=B043_SAMPLE_LIMIT,
            sample_budget_seconds=30.,require_graph_smoke=True)
        if not GPU_REPORT['graph_smoke_passed'] or GPU_REPORT['status']!='COMPLETE':
            GPU_REPORT['status']='FAILED'
            GPU_REPORT['reason']='Initial real-GPU smoke had no graph READ; inspect saved traces before a larger run'
    except Exception as exc:
        import traceback
        (OUTPUT/'gpu_failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
        GPU_REPORT={'status':'FAILED','gpu_executed':False,'error':str(exc),'reason':'Partial checkpoints and traceback saved'}
write_json(OUTPUT/'gpu_status.json',GPU_REPORT)
print({k:GPU_REPORT.get(k) for k in ['status','gpu_executed','n_runs','graph_smoke_passed','reason']})


In [ ]:
import shutil
write_json(OUTPUT/'delivery_manifest.json',{'cpu_status':CPU_REPORT['status'],'gpu_status':GPU_REPORT['status'],
    'payload_sha256':PAYLOAD_SHA256,'compiled_payload_sha256':compiled['payload_sha256'],
    'privacy':'Private development predictions/traces; do not publish without project permission'})
ZIP_PATH=Path(shutil.make_archive(str(B043_BASE/'LifeMate_B043_final_outputs'),'zip',OUTPUT))
print('Download:',ZIP_PATH)
try:
    from IPython.display import FileLink,display
    display(FileLink(str(ZIP_PATH)))
except ImportError:pass
if GPU_REPORT['status']=='FAILED':raise RuntimeError('GPU stage failed; output ZIP and checkpoints are available')


## Outputs and interpretation

`benchmark/benchmark_report.json`, `run_manifest.json`, `predictions.jsonl`, `traces_PRIVATE.jsonl` and `checkpoints/` contain actual GPU runs only. Initial smoke must show graph_reads > 0 before the representative run continues. Selection includes single-tool, multi-tool/app/hop, ambiguous, missing-info, no-op and READ→WRITE strata; it is train-development, not an independent holdout.

Both arms use the same model, seed, ToolEnv and bounded inference settings. One ToolEnv is used per trajectory; no hidden state, future observations or gold rollout is used. Labels are evaluated only after each trajectory. `state_hash` is saved; State Match stays null without independently supplied full-state reference hashes. The fuzzy-call metric is explicitly a proxy until the official evaluator/tokenizer is available. Unnecessary READ, wrong-subject and unsafe-WRITE semantic rates stay null without independent adjudication.

30s is enforced before generation/new tool calls, with a remaining-time generation limit. CUDA passes and ToolEnv calls cannot be forcibly preempted; actual overruns are counted. Model loading/installation are outside per-sample inference timing. GRAPH_DISABLED remains the preserved B042 execution path with the same benchmark budget.

Resume only with the same fingerprint; changed source/model/graph/settings require a new B043_WORKDIR. No parity-92, Shadow benchmark, fine-tuning or graph discovery runs here.
